In [1]:
from huggingface_hub import login
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

In [2]:
load_dotenv()

True

In [3]:
login()

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Embdedding documents

In [5]:
from ingest import load_faq_data

In [6]:
documents = load_faq_data()

In [7]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [9]:
from tqdm.auto import tqdm

In [10]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

In [14]:
len(vectors)

1346

## Vector search with sqlitesearch

In [2]:
from sqlitesearch import VectorSearchIndex

In [3]:
vindex = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors.db'
)

In [15]:
vindex.fit(vectors, documents)

In [16]:
query = "How can I get a certificate ?"
query_vector = model.encode(query)

In [19]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [20]:
print(results)

[{'id': '69d122f12e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?', 'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'}, {'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '9f689c185f', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I missed the first homework - can I stil

In [4]:
vindex.close()

## Reopening the index

In [22]:
vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode='ivf',
    db_path="faq_vectors.db"
)

In [23]:
query = "How do I run Kafka?"
query_vector = model.encode(query)

In [24]:
results = vs_index.search(
    query_vector,
    num_results=5
)

In [31]:
for doc in results:
    print(doc['question'])
    print(doc['answer'])
    print("**************************************")

Java Kafka: How to run producer/consumer/kstreams/etc in terminal
In the project directory, run:

```bash
java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java
```
**************************************
Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent
For example, when running `JsonConsumer.java`, you might see:

```
Consuming form kafka started

RESULTS:::0

RESULTS:::0

RESULTS:::0
```

Or when running `JsonProducer.java`, you might encounter:

```
Exception in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed
```

**Solution:**

1. Ensure the `StreamsConfig.BOOTSTRAP_SERVERS_CONFIG` in the scripts located at `src/main/java/org/example/` (e.g., `JsonConsumer.java`, `JsonProducer.java`) is pointing to the correct server URL (e.g., `europe-west3` vs `europe-west2`).

2. Verify that the cluster key and se

In [ ]:
vs_index.close()

In [32]:
import anthropic
from rag_helper import RAGVector

In [33]:
client = anthropic.Anthropic()

In [40]:
vector_assistant = RAGVector(
    index=vs_index,
    llm_client=client,
    embedder=model,
    course="data-engineering-zoomcamp",
    model="claude-haiku-4-5"
)

In [41]:
results = vector_assistant.rag("How do I run Kafka?")

In [42]:
print(results.content[0].text)

# How to Run Kafka

Based on the context, here are the main ways to run Kafka:

## For Java Applications

Run the following command in your project directory:

```bash
java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java
```

Replace `JsonProducer.java` with the specific Kafka component you want to run (e.g., `JsonConsumer.java`, `KStreams`, etc.).

## For Python Applications

1. **Create and activate a virtual environment:**
   - macOS/Linux: `source env/bin/activate`
   - Windows: `env\Scripts\activate`

2. **Install required packages:**
   ```bash
   pip install -r requirements.txt
   ```

3. **Ensure Docker is running** before executing your Python file

## Important Prerequisites

Before running Kafka, make sure:
- Your Kafka broker Docker container is running
- If it's not running, use: `docker-compose up -d`
- Verify containers are running with: `docker ps`
- Your configuration credentials (bootstrap server URL, cluster key, and secret)